# 1. Merge & Pre-processing
The cleaned transactions (`train`) and the cleaned customer table (`clientes`) are merged into one dataset, split into training and validation folds, and pre-processed in order: outliers, missing values, and encoding and scaling.

In [10]:
import warnings
from collections import namedtuple
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (needed for IterativeImputer)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

# Same seed everywhere, so every run gives the same splits and draws
RANDOM_STATE = 42

____________________________________________________________________________________________________

## 1.1. Data Loading

In [ ]:
trainFile = "../../data/trainDataset/train_2_5_1.csv"
clientFile = "../../data/clientesDataset/clientes_clean.csv"
testFile = "../../data/test.csv"

train = pd.read_csv(trainFile)
clientes = pd.read_csv(clientFile)
test = pd.read_csv(testFile)

RANDOM_STATE= 42

____________________________________________________________________________________________________

## 1.2. Merging the Datasets

In [12]:
train_clientes = train.merge(clientes, on="customer_id", how="left", validate="many_to_one")
assert len(train_clientes) == len(train)

train_clientes.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,island,municipality,city,postal_area,postal_code,street,address,company_name,customer_since_year,since_2015_or_earlier
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,(mainland),Municipality MU89: Council of Last Minute Edits,City CY286: Cached Result Falls,Postal area PA228: Hidden Sheet Sector,Postal code PC444: #REF! Delivery Route,Street ST303: Absolute Reference Terrace,Address AD761: Cell Without a Home,Excel Is Not A Database Ltd 1013,2024,0
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,City CY147: VLOOKUP Bay,Postal area PA260: Hidden Sheet Sector,Postal code PC636: #REF! Delivery Route,NaN,Address AD382: Cell Without a Home,Excel Is Not A Database Ltd 35,2015,1
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,(mainland),Municipality MU70: Council of Last Minute Edits,City CY265: Where-less Query Falls,Postal area PA84: Hidden Sheet Sector,Postal code PC200: #REF! Delivery Route,NaN,Address AD723: Cell Without a Home,Excel Is Not A Database Ltd 826,2017,0
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,NaN,Postal area PA253: Hidden Sheet Sector,Postal code PC558: #REF! Delivery Route,Street ST144: Last Save Court,Address AD399: Cell Without a Home,Excel Is Not A Database Ltd 530,2018,0
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,(mainland),Municipality MU142: Council of Last Minute Edits,City CY359: Spill Error Ridge,Postal area PA136: Hidden Sheet Sector,Postal code PC300: #REF! Delivery Route,Street ST395: AutoFill Path,Address AD928: Cell Without a Home,Excel Is Not A Database Ltd 220,2017,0


**Conclusions:**
* **Lossless merge:** every one of the **6,469** transactions found its customer (`validate="many_to_one"` confirms one row per `customer_id` in `clientes`), so no rows were lost or duplicated. The merged table has **57 columns**: the 45 of `train` plus 12 customer attributes.
* **Customer coverage:** the transactions belong to **607** customers, so 536 of the 1,143 customers in `clientes` have no purchase in `train` and are simply not used.
* **New missing values:** the customer attributes bring their own gaps into the merged table, mostly in identifier-like columns: `street` (58.9%), `city` (32.9%) and `postal_code` (3.4%). The descriptive ones are almost complete: `postal_area` (1.0%), `commercial_region` (0.5%), `island` and `municipality` (0.1%). These are handled together with the `train` gaps in the next sections.

____________________________________________________________________________________________________

## 1.3. Feature Type Separation
Imputation and encoding work differently on numbers and on labels (a median or a KNN fill needs numbers, a mode or an `"Unknown"` level works on labels), so the merged columns are split into groups first:

1. **Target:** `repeat_purchase_90d`, the variable to predict.
2. **Identifiers:** `ID` (one per row) and `customer_id` (the join key, kept to group the cross-validation folds by customer).
3. **Excluded:** `company_name`, `address`, `street` and `postal_code` are distinct for almost every customer, so they name a customer instead of describing one; `random_noise` carries no information by construction.
4. **Numeric** and **categorical** features: everything else, separated by data type.

In [13]:
target = "repeat_purchase_90d"
identifiers = ["ID", "customer_id"]

# Unique per customer (they name a customer instead of describing one) or pure noise
excluded = ["company_name", "address", "street", "postal_code", "random_noise"]

features = [c for c in train_clientes.columns if c not in [target] + identifiers + excluded]

# Automatic separation between numerical and categorical variables
numeric = [c for c in features if pd.api.types.is_numeric_dtype(train_clientes[c])]
categorical = [c for c in features if c not in numeric]

# Every column belongs to exactly one group
assert set([target] + identifiers + excluded + numeric + categorical) == set(train_clientes.columns)

features = train_clientes[numeric + categorical]
identifier = train_clientes[target]

print(f"{len(numeric)} numeric | {len(categorical)} categorical | {len(identifiers)} identifiers | {len(excluded)} excluded")
print("categorical:", categorical)

37 numeric | 12 categorical | 2 identifiers | 5 excluded
categorical: ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'country_or_region', 'commercial_region', 'island', 'municipality', 'city', 'postal_area']


**Conclusions:**
* **49 features** are left for the model: **37 numeric** and **12 categorical**. The target has no missing values and is fairly balanced (56% of purchases are repeated within 90 days).
* **Numbers that are labels:** some numeric columns are categories written as numbers: `purchase_month`, `purchase_dayofweek`, `purchase_quarter` and the 0/1 flags `is_first_purchase`, `is_weekend` and `since_2015_or_earlier`. None of them has missing values, so they do not affect the imputation; whether they are encoded as categories is decided in the encoding step.

____________________________________________________________________________________________________

## 1.4. Missing Values Overview

In this step, we measure the missing values, which gave us some information about which features still have gaps after the merge, how many, and whether if a value is missing says something about the target.

In [14]:
# Which features still have missing values, and in which group
missing = features.isna().sum()
overview = pd.DataFrame({
    "group": ["numeric" if c in numeric else "categorical" for c in features.columns],
    "n_missing": missing,
    "pct_missing": (missing / len(features) * 100).round(1),
    # Repeat rate among rows where the value is missing, against rows where it is present
    "target_if_missing": [identifier[features[c].isna()].mean() for c in features.columns],
    "target_if_present": [identifier[features[c].notna()].mean() for c in features.columns],
}).query("n_missing > 0").sort_values("n_missing", ascending=False).round(3)

print(f"rows with at least one missing value: {features.isna().any(axis=1).sum():,} of {len(features):,}")
overview

rows with at least one missing value: 2,823 of 6,469


,group,n_missing,pct_missing,target_if_missing,target_if_present
city,categorical,2128,32.9,0.553,0.563
spend_365d,numeric,210,3.2,0.552,0.560
transport_method,categorical,196,3.0,0.551,0.560
payment_type,categorical,195,3.0,0.564,0.560
warehouse,categorical,195,3.0,0.636,0.557
salesperson,categorical,194,3.0,0.598,0.559
postal_area,categorical,65,1.0,0.492,0.560
commercial_region,categorical,35,0.5,0.343,0.561
municipality,categorical,4,0.1,0.000,0.560
island,categorical,4,0.1,0.000,0.560


**Conclusions:**

____________________________________________________________________________________________________

## 1.5. Train / Validation Split

### 1.5.1. Evaluation Setup
Every comparison below runs through the same three tools, written once, as in the course practical: the **model matrix** (`design_matrix`), the **fill** that makes a split ready for a model (`fill_missing`), and the **scorer** (`held_out_f1`), which trains a logistic regression on the training half of each split and returns its F1 on the untouched validation half.

In [15]:
# The table every comparison works on: the features plus the target
frame = features.join(identifier)

# Logistic regression may stop before converging on unscaled columns; scaling is compared in 1.8
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def design_matrix(frame, numeric, categorical):
    """Numbers as they are, everything else one-hot. Identifiers never enter."""
    encoder = OneHotEncoder(sparse_output=False, dtype=float)
    encoded = encoder.fit_transform(frame[categorical].astype("string"))
    text = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(categorical),
        index=frame.index,
    )
    return pd.concat([frame[numeric], text], axis=1)


def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs: each imputer is fitted on the
    training rows of its own columns and fills those columns in both halves, so
    the validation rows never help compute a fill.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # scikit-learn reads np.nan as a gap but not pd.NA, so every gap becomes np.nan
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        if isinstance(imputer, (KNNImputer, IterativeImputer)):
            # Both are sensitive to scale, so they work on standardised columns
            # and their fills go back in the columns' own units
            scale = StandardScaler().fit(blocks[0])
            imputer.fit(scale.transform(blocks[0]))
            filled = [scale.inverse_transform(imputer.transform(scale.transform(block)))
                      for block in blocks]
        else:
            imputer.fit(blocks[0])
            filled = [imputer.transform(block) for block in blocks]
        train[columns], test[columns] = filled
    return train, test


# A model cannot fit a missing value, so every comparison fills inside its own split.
# Until section 1.7 chooses a plan, that fill is the training median for the numbers
# and the training mode for the labels.
median_mode = [(numeric, SimpleImputer(strategy="median")),
               (categorical, SimpleImputer(strategy="most_frequent"))]

Scored = namedtuple("Scored", "f1 iterations")


def sem(values):
    """Standard error of the mean: how far it would move on new splits."""
    return values.std(ddof=1) / np.sqrt(len(values))


def held_out_f1(prepare, frame, splits, numeric, categorical,
                target="repeat_purchase_90d", scaler=None):
    """Score one treatment by F1 on each split's untouched validation rows.

    `prepare(train, test)` applies the treatment, fitted on `train` only. The
    result holds the validation F1 per split (`.f1`) and the solver's iterations
    (`.iterations`). `scaler`, when given, is fitted on the training half and
    applied to both.
    """
    f1, iterations = [], []
    for train_rows, test_rows in splits:
        train, test = prepare(frame.iloc[train_rows], frame.iloc[test_rows])
        matrix = design_matrix(pd.concat([train, test]), numeric, categorical)
        a, b = matrix.iloc[:len(train)], matrix.iloc[len(train):]
        if scaler is not None:
            fitted = scaler.fit(a)
            a, b = fitted.transform(a), fitted.transform(b)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(a, train[target])
        iterations.append(int(model.n_iter_[0]))
        f1.append(f1_score(test[target], model.predict(b)))
    return Scored(np.array(f1), np.array(iterations))

### 1.5.2. The Splits
Following the course practical (`week_03_deepen_exploration`), instead of a single train/validation split the data is divided **20 times** with `StratifiedShuffleSplit` (80% training, 20% validation, `random_state=42`), and every candidate treatment is scored on the **same 20 validation sets**. A difference between two treatments therefore comes from the treatments, not from a lucky split, and it can be judged by its mean F1 and by how many of the 20 splits it wins.

* **Stratified:** each split keeps the class balance of the target `repeat_purchase_90d`, so training and validation have the same share of repeat purchases.
* **Fitted on training rows only:** every step that learns from the data (outlier limits, imputers, encoders, scalers) is fitted on the 80% training rows of each split and only applied to its 20% validation rows. A median computed over validation rows would let the validation set leak into training.
* **Rows are never dropped from validation:** a treatment may remove rows from the training side only, so every candidate is scored on identical, untouched validation rows.
* **Drawn once:** the 20 pairs are drawn once on `frame` (`frame_splits`) and reused by sections 1.6, 1.7 and 1.8. A treatment that drops rows does it inside a split, on its training half, so the pairs stay valid.

In [16]:
# Twenty independent 80/20 draws that keep the class balance of the target, so every
# candidate below is judged on the same twenty validation sets
splits = StratifiedShuffleSplit(n_splits=20, test_size=0.2,
                                random_state=RANDOM_STATE)

# The 20 (training rows, validation rows) pairs, reused by every comparison below
frame_splits = list(splits.split(frame, frame["repeat_purchase_90d"]))

In [17]:
# Check: every split has the same size and keeps the target rate of the full table
pd.DataFrame([{"training rows": len(tr), "validation rows": len(va),
               "training rate": identifier.iloc[tr].mean(), "validation rate": identifier.iloc[va].mean()}
              for tr, va in frame_splits]).describe().loc[["mean", "min", "max"]].round(3)

,training rows,validation rows,training rate,validation rate
mean,5175.0,1294.0,0.56,0.56
min,5175.0,1294.0,0.56,0.56
max,5175.0,1294.0,0.56,0.56


The **baseline** every later section has to beat: no treatment, only the default median/mode fill inside each split.

In [18]:
baseline = held_out_f1(partial(fill_missing, plan=median_mode),
                       frame, frame_splits, numeric, categorical)

print(f"baseline F1 {baseline.f1.mean():.4f} +/- {sem(baseline.f1):.4f}"
      f"   (min {baseline.f1.min():.4f}, max {baseline.f1.max():.4f})")

baseline F1 0.6803 +/- 0.0032   (min 0.6454, max 0.7042)


**Conclusions:**
* **The splits hold:** each of the 20 splits has **5,175 training rows** and **1,294 validation rows**, and the share of repeat purchases is **0.56 in both**, the same as the full table, so the stratification works.
* **Baseline:** with no treatment and only the median/mode fill, the logistic regression reaches **F1 = 0.6803 ± 0.0032** (between 0.6454 and 0.7042 across the splits). Every treatment in sections 1.6 to 1.8 is compared against this value, on the same 20 splits.
* **Not converged:** on these unscaled columns the solver stops at its 1,000-iteration cap on every split. Scaling, compared in section 1.8, is expected to fix this.
* **Limitation:** `test.csv` covers a later period (May 2025 to October 2026) than `train` (July 2015 to February 2025). A random split mixes future purchases into training, so these F1 scores are optimistic. They are used to **compare** treatments with each other, not to estimate the score on the test set.

### 1.5.3. Temporal Splits
`test.csv` covers a later period than `train` (May 2025 to October 2026 against July 2015 to February 2025), and the competition rules penalise a large gap between the performance a notebook reports and its Kaggle result. The random splits above mix future purchases into training, so they are kept to **compare** treatments, and a second set of splits is drawn **in time** to confirm the final recipe and to report the final model's performance.

* **Rolling origin:** the rows are ordered by purchase date and cut into 5 consecutive validation blocks with `TimeSeriesSplit`; each block is validated by a model trained only on the purchases before it.
* **90-day gap:** the target looks 90 days ahead, so the training purchases from the 90 days before each validation block are left out. Their labels depend on purchases inside the block, and the same gap separates `train` from `test.csv` (98 days).
* **Same scorer:** `held_out_f1` accepts any list of (training rows, validation rows) pairs, so these splits use it unchanged.

**Conclusions:**

____________________________________________________________________________________________________

## 1.6. Missing Values Comparison and Selection

In [19]:
class ProportionalImputer:
    """Fill each missing label with one drawn from its column's training labels,
    in their training shares.

    scikit-learn has no such imputer, and `fill_missing` needs only `fit` and
    `transform`, so a short class is enough.
    """

    def __init__(self, random_state=None):
        self.random_state = random_state

    def fit(self, frame):
        self.shares = {c: frame[c].value_counts(normalize=True) for c in frame}
        return self

    def transform(self, frame):
        draws = np.random.default_rng(self.random_state)
        frame = frame.copy()
        for column, shares in self.shares.items():
            gaps = frame[column].isna()
            if gaps.any():
                frame.loc[gaps, column] = draws.choice(
                    shares.index, size=gaps.sum(), p=shares.to_numpy())
        return frame



### 1.6.1. Fill Plans

In this sub section we define the possible inputers as well as the plans. After defining we call on held_out_f1 function 

In [ ]:
# One imputer per fill. "own level" gives a missing label a category of its
# own instead of the common one.
fills = {"median": SimpleImputer(strategy="median"),
         "mean": SimpleImputer(strategy="mean"),
         "zero": SimpleImputer(strategy="constant", fill_value=0),
         "mode": SimpleImputer(strategy="most_frequent"),
         "own level": SimpleImputer(strategy="constant", fill_value="(missing)"),
         "proportional": ProportionalImputer(random_state=RANDOM_STATE)}

plans = {
    "median / mode": [(numeric, fills["median"]), (categorical, fills["mode"])],
    "mean / mode": [(numeric, fills["mean"]), (categorical, fills["mode"])],
    "zero / mode": [(numeric, fills["zero"]), (categorical, fills["mode"])],
    "median / own level": [(numeric, fills["median"]), (categorical, fills["own level"])],
    "median / proportional": [(numeric, fills["median"]), (categorical, fills["proportional"])],
}

plan_scores = {name: held_out_f1(partial(fill_missing, plan=plan),
                                 frame, frame_splits, numeric, categorical)
               for name, plan in plans.items()}


def fill_with_indicator(train, test, plan):
    """Mark where spend_365d was missing in its own 0/1 column, then fill."""
    train, test = (f.assign(spend_365d_missing=f["spend_365d"].isna().astype(int))
                   for f in (train, test))
    return fill_missing(train, test, plan)


# The indicator arm adds a column, so its scorer needs the longer numeric list
plan_scores["median + indicator / mode"] = held_out_f1(
    partial(fill_with_indicator, plan=plans["median / mode"]),
    frame, frame_splits, numeric + ["spend_36No teu caso: na 1.4 viste que a taxa de recompra é quase igual com e sem spend_365d (0,552 contra 0,560). Isso sugere que a falha não diz grande coisa sobre o target, por isso é provável que o indicador não ajude. Mas só a comparação nas 20 divisões o confirma.5d_missing"], categorical)

plan_board = pd.DataFrame({
    "F1": {k: v.f1.mean() for k, v in plan_scores.items()},
    "F1 SEM": {k: sem(v.f1) for k, v in plan_scores.items()},
})
print(plan_board.round(4).to_string())

                               F1  F1 SEM
median / mode              0.6803  0.0032
mean / mode                0.6818  0.0029
zero / mode                0.6817  0.0030
median / own level         0.6813  0.0028
median / proportional      0.6841  0.0027
median + indicator / mode  0.6793  0.0032


### 1.6.2. Paired Comparison

In [21]:
shipped = plan_scores["median / mode"]

# Paired: each arm minus the shipped arm on the SAME split, so the noise they
# share cancels and what is left is the difference between the arms.
paired = pd.DataFrame({
    name: {"dF1": (run.f1 - shipped.f1).mean(),
           "F1 SEM": sem(run.f1 - shipped.f1)}
    for name, run in plan_scores.items() if name != "median / mode"}).T
print(paired.round(4).to_string())

                              dF1  F1 SEM
mean / mode                0.0015  0.0020
zero / mode                0.0014  0.0018
median / own level         0.0010  0.0021
median / proportional      0.0038  0.0021
median + indicator / mode -0.0010  0.0020


### 1.6.3. Final Imputation
The plans above all fill the gaps. The last comparison asks whether filling is better than simply dropping the training rows that carry a missing value. Rows are dropped from the training half only, so both options are scored on the same untouched validation rows.

In [22]:
def drop_incomplete(train, test, plan):
    """Drop the training rows carrying any missing value, then fill what the rest need."""
    complete = ~train.isna().any(axis=1)
    return fill_missing(train[complete], test, plan)


keep_rows = plan_scores["median / mode"].f1
drop_rows = held_out_f1(partial(drop_incomplete, plan=plans["median / mode"]),
                        frame, frame_splits, numeric, categorical).f1
paired = keep_rows - drop_rows

print(f"impute the missing values  n={len(frame):,}   F1 {keep_rows.mean():.4f}")
print(f"drop the rows    n={(~frame.isna().any(axis=1)).sum():,}"
      f"   F1 {drop_rows.mean():.4f}")
print(f"paired difference {paired.mean():+.4f}"
      f" +/- {sem(paired):.4f}"
      f"   imputing wins on {(paired > 0).sum()} of {len(paired)} splits")

impute the missing values  n=6,469   F1 0.6803
drop the rows    n=3,646   F1 0.6538
paired difference +0.0265 +/- 0.0021   imputing wins on 20 of 20 splits


In [23]:
# The fill plan this section chose
chosen = "median / mode"

**Conclusions:**
* **No fill plan beats `median / mode` by more than two paired standard errors.** `median / proportional` comes closest (+0.0038 ± 0.0021, 1.8 SEM); `mean / mode` (+0.0015 ± 0.0020), `zero / mode` (+0.0014 ± 0.0018) and `median / own level` (+0.0010 ± 0.0021) are all within one. The differences are small because only `spend_365d` has numeric gaps (3.2%), so the numeric fill changes a single column.
* **The missingness indicator does not help** (−0.0010 ± 0.0020), which matches section 1.4: a missing `spend_365d` barely changes the repeat rate (0.552 against 0.560), so the extra column carries no signal.
* **Filling is clearly better than dropping:** dropping the incomplete training rows keeps only 3,646 of the 6,469 rows (44% lost, mostly because of `city`) and scores 0.6538 against 0.6803, a paired difference of +0.0265 ± 0.0021 in favour of filling, ahead on all 20 splits.
* **The recipe keeps `median / mode`**, the simplest plan: it adds no random draws and no extra columns, and no alternative earns its extra complexity. `KNN` and `iterative` imputation were not scored: only one numeric column has gaps, and in the course practical they brought no gain over the median.

____________________________________________________________________________________________________

## 1.7. Outliers

### 1.7.1. Outlier Detection

In [ ]:
def outside_fences(frame, columns, rule="iqr", k=1.5):
    """True for rows with a named column beyond its fences.

    `rule="iqr"` draws the fences k IQRs beyond the quartiles, `rule="zscore"`
    k standard deviations from the mean. A missing value counts as INSIDE: a
    comparison with a missing value is False both ways, so without the `isna()`
    term every row carrying one would be called an outlier.
    """
    values = frame[columns]
    if rule == "iqr":
        q1, q3 = values.quantile(0.25), values.quantile(0.75)
        low, high = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
    else:
        mean, sd = values.mean(), values.std(ddof=0)
        low, high = mean - k * sd, mean + k * sd
    within = values.ge(low) & values.le(high)
    return ~(within | values.isna()).all(axis=1)

### 1.7.2. Outlier Treatment

**Conclusions:**

____________________________________________________________________________________________________

## 1.8. Encoding and Scaling

### 1.8.1. Encoding

### 1.8.2. Scaling

**Conclusions:**